# 🌿 Plant Disease Detection using CNN
## Notebook 3 — Web App & Real-World Field Test (Phase 3)

### What this notebook does
1. Loads the trained models from Google Drive (made in Notebook 2).
2. **Part A — Web app:** a simple page where you upload a leaf photo and get the disease name, confidence and top-3 guesses.
3. **Part B — Real-world test:** checks how the models perform on **PlantDoc**, real photos taken in fields with messy backgrounds, to measure the gap between lab and field.

Run Notebook 2 first. Then `Runtime → Change runtime type → T4 GPU` and run the cells from top to bottom.

In [ ]:
!pip -q install gradio

In [ ]:
# ============================================================
# SETUP: imports, settings and project folders
# ============================================================
import os, sys, json, time, random, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42                     # fixed "random seed" -> same results every run
random.seed(SEED); np.random.seed(SEED)

IMG_SIZE = 224                # every image is resized to 224 x 224 pixels
BATCH_SIZE = 32               # how many images the model sees at once

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # Google Drive keeps our trained models & results safe even if Colab disconnects
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        PROJECT_DIR = '/content/drive/MyDrive/plant-disease-project'
    except Exception as e:
        # Drive permission failed (tip: tick "Select all" on the Google permission screen).
        # We continue anyway, but results are only kept until this Colab session ends.
        print('WARNING: Google Drive could not be connected:', e)
        print('Saving to /content/outputs instead - download the results before closing Colab!')
        PROJECT_DIR = '/content/outputs'
    DATA_ROOT = '/content/data'              # fast local disk for the images
else:
    PROJECT_DIR = os.path.abspath('../outputs')
    DATA_ROOT = os.path.abspath('../data')

MODELS_DIR  = os.path.join(PROJECT_DIR, 'models')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FIG_DIR     = os.path.join(RESULTS_DIR, 'figures')
for d in [MODELS_DIR, RESULTS_DIR, FIG_DIR, DATA_ROOT]:
    os.makedirs(d, exist_ok=True)

DATASET_DIR = os.path.join(DATA_ROOT, 'PlantVillage-Dataset', 'raw', 'color')
print('Running in Colab:', IN_COLAB)
print('Results will be saved in:', PROJECT_DIR)

In [ ]:
# ============================================================
# HELPER: turn folder names into readable names
# 'Tomato___Early_blight'  ->  'Tomato - Early blight'
# ============================================================
def pretty_name(folder_name):
    crop, disease = folder_name.split('___')
    crop = crop.replace('_', ' ').replace(',', '').strip()
    disease = disease.replace('_', ' ').strip()
    disease = disease[0].upper() + disease[1:]
    return f'{crop} - {disease}'

def crop_of(folder_name):
    return folder_name.split('___')[0].replace('_', ' ').replace(',', '').strip()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from PIL import Image

with open(os.path.join(MODELS_DIR, 'class_names.json')) as f:
    CLASS_NAMES = json.load(f)
NUM_CLASSES = len(CLASS_NAMES)
SHORT = [pretty_name(c) for c in CLASS_NAMES]

models = {}
for key, file in [('MobileNetV2', 'mobilenetv2.keras'), ('Basic CNN', 'basic_cnn.keras')]:
    path = os.path.join(MODELS_DIR, file)
    if os.path.exists(path):
        models[key] = keras.models.load_model(path)
        print(f'Loaded {key} from {path}')
assert models, 'No trained models found - please run Notebook 2 first.'

def preprocess(pil_img):
    """Same preprocessing as training: RGB, resize to 224x224, values 0-255.
    (Normalisation happens inside the model.)"""
    img = pil_img.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)[None, ...]

## Part A — Web app (Gradio)
Running the next cell starts the app. A **public link** (`https://xxxx.gradio.live`) appears, which you can open on your phone too. It stays valid for 72 hours while this notebook is running.

> ⚠️ The app is a study project, not a replacement for an agricultural expert.

In [ ]:
import gradio as gr

def predict(pil_img, model_name):
    if pil_img is None:
        return {}, 'Please upload a leaf image.'
    probs = models[model_name].predict(preprocess(pil_img), verbose=0)[0]
    top = int(probs.argmax())
    status = '✅ Healthy' if 'healthy' in CLASS_NAMES[top] else '⚠️ Disease detected'
    msg = (f'### {status}\n**Prediction:** {SHORT[top]}  \n**Confidence:** {probs[top]*100:.1f}%')
    if probs[top] < 0.5:
        msg += '\n\n_Low confidence: try a clearer photo of a single leaf on a plain background._'
    return {SHORT[i]: float(probs[i]) for i in range(NUM_CLASSES)}, msg

EX_DIR = os.path.join(PROJECT_DIR, 'examples')
examples = [[os.path.join(EX_DIR, f), list(models)[0]] for f in sorted(os.listdir(EX_DIR))] if os.path.isdir(EX_DIR) else None

demo = gr.Interface(
    fn=predict,
    inputs=[gr.Image(type='pil', label='Upload a leaf photo'),
            gr.Radio(list(models), value=list(models)[0], label='Model')],
    outputs=[gr.Label(num_top_classes=3, label='Top-3 predictions'), gr.Markdown()],
    title='🌿 Plant Disease Detection (CNN)',
    description='Upload a photo of a single leaf (apple, blueberry, cherry, corn, grape, orange, peach, '
                'pepper, potato, raspberry, soybean, squash, strawberry or tomato). '
                'The model predicts one of 38 healthy/disease classes.',
    examples=examples,
)
demo.launch(share=True, debug=False)

## Part B — Real-world test on PlantDoc field images
**PlantDoc** (Singh et al., 2020, IIT Gandhinagar) contains leaf photos downloaded from the internet: real fields, hands, multiple leaves, shadows. All 28 of its classification folders correspond to a PlantVillage class, so we can test our models on them **without any retraining**.

This answers an important question for the report: *does high accuracy on lab photos carry over to real farm photos?*

**Citation:** Singh, D., Jain, N., Jain, P., Kayal, P., Kumawat, S., & Batra, N. (2020). *PlantDoc: A Dataset for Visual Plant Disease Detection.* CoDS-COMAD 2020. arXiv:1911.10317.

In [ ]:
# PlantDoc folder name  ->  matching PlantVillage class
PLANTDOC_TO_PV = {
    'Apple Scab Leaf':            'Apple___Apple_scab',
    'Apple leaf':                 'Apple___healthy',
    'Apple rust leaf':            'Apple___Cedar_apple_rust',
    'Bell_pepper leaf':           'Pepper,_bell___healthy',
    'Bell_pepper leaf spot':      'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf':             'Blueberry___healthy',
    'Cherry leaf':                'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot':        'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight':           'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf':             'Corn_(maize)___Common_rust_',
    'Peach leaf':                 'Peach___healthy',
    'Potato leaf early blight':   'Potato___Early_blight',
    'Potato leaf late blight':    'Potato___Late_blight',
    'Raspberry leaf':             'Raspberry___healthy',
    'Soyabean leaf':              'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf':            'Strawberry___healthy',
    'Tomato Early blight leaf':   'Tomato___Early_blight',
    'Tomato Septoria leaf spot':  'Tomato___Septoria_leaf_spot',
    'Tomato leaf':                'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight':    'Tomato___Late_blight',
    'Tomato leaf mosaic virus':   'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus':   'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf':           'Tomato___Leaf_Mold',
    'Tomato two spotted spider mites leaf': 'Tomato___Spider_mites Two-spotted_spider_mite',
    'grape leaf':                 'Grape___healthy',
    'grape leaf black rot':       'Grape___Black_rot',
}

In [ ]:
# Download PlantDoc (small: ~2,600 images). We use BOTH its train and test folders, since we never train on it.
PD_DIR = os.path.join(DATA_ROOT, 'PlantDoc-Dataset')
if not os.path.isdir(PD_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pratikkayal/PlantDoc-Dataset', PD_DIR], check=True)

In [ ]:
pd_records = []
for split in ['train', 'test']:
    if not os.path.isdir(os.path.join(PD_DIR, split)):
        continue
    for folder in sorted(os.listdir(os.path.join(PD_DIR, split))):
        if folder not in PLANTDOC_TO_PV:
            continue
        for fn in sorted(os.listdir(os.path.join(PD_DIR, split, folder))):
            pd_records.append((os.path.join(split, folder, fn), folder, CLASS_NAMES.index(PLANTDOC_TO_PV[folder])))
plantdoc_df = pd.DataFrame(pd_records, columns=['filepath', 'plantdoc_class', 'label_id'])
print(f'{len(plantdoc_df)} PlantDoc images in {plantdoc_df.plantdoc_class.nunique()} classes')

In [ ]:
# Load the images (some web images are broken or unusual formats - we skip those)
pd_images, keep = [], []
for i, p in enumerate(plantdoc_df['filepath']):
    try:
        with Image.open(os.path.join(PD_DIR, p)) as im:
            pd_images.append(preprocess(im)[0]); keep.append(i)
    except Exception:
        pass
plantdoc_df = plantdoc_df.iloc[keep].reset_index(drop=True)
pd_images = np.stack(pd_images)
print(f'{len(plantdoc_df)} images loaded successfully')

field_probs = {name: m.predict(pd_images, batch_size=32, verbose=1) for name, m in models.items()}

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

y_field = plantdoc_df['label_id'].values
field_crop = np.array([crop_of(CLASS_NAMES[i]) for i in y_field])
lab = pd.read_csv(os.path.join(RESULTS_DIR, 'model_comparison.csv'), index_col=0)

rows = []
for name, probs in field_probs.items():
    pred = probs.argmax(1)
    pred_crop = np.array([crop_of(CLASS_NAMES[i]) for i in pred])
    lab_name = [m for m in lab.index if m.startswith(name)]
    rows.append({
        'Model': name,
        'Lab test accuracy (%) - PlantVillage': lab.loc[lab_name[0], 'Test accuracy (%)'] if lab_name else np.nan,
        'Field accuracy (%) - PlantDoc': accuracy_score(y_field, pred) * 100,
        'Field top-3 accuracy (%)': top_k_accuracy_score(y_field, probs, k=3, labels=np.arange(NUM_CLASSES)) * 100,
        'Field macro F1 (%)': f1_score(y_field, pred, average='macro', labels=np.unique(y_field), zero_division=0) * 100,
        'Correct crop identified (%)': (pred_crop == field_crop).mean() * 100,
    })
field_results = pd.DataFrame(rows).set_index('Model')
field_results.to_csv(os.path.join(RESULTS_DIR, 'field_test_plantdoc.csv'))
field_results.round(2)

In [ ]:
# Visual comparison: lab vs field accuracy
ax = field_results[['Lab test accuracy (%) - PlantVillage', 'Field accuracy (%) - PlantDoc']].plot(
        kind='bar', figsize=(8, 4.5), color=['#2e7d32', '#ef6c00'], rot=0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f', padding=2)
plt.ylabel('Accuracy (%)'); plt.ylim(0, 105); plt.title('Lab photos vs real field photos')
plt.legend(['Lab (PlantVillage test set)', 'Field (PlantDoc)'], loc='upper right')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, 'lab_vs_field_accuracy.png'), dpi=150); plt.show()

In [ ]:
# Some field predictions from the best model
best = list(models)[0]
probs = field_probs[best]; pred = probs.argmax(1)
rng = np.random.default_rng(SEED)
idx = rng.choice(len(plantdoc_df), min(12, len(plantdoc_df)), replace=False)
plt.figure(figsize=(16, 11))
for k, i in enumerate(idx):
    with Image.open(os.path.join(PD_DIR, plantdoc_df['filepath'].iloc[i])) as im:
        img = im.convert('RGB')
    ok = pred[i] == y_field[i]
    plt.subplot(3, 4, k + 1); plt.imshow(img); plt.axis('off')
    plt.title(f'Pred: {SHORT[pred[i]]} ({probs[i].max()*100:.0f}%)\nTrue: {SHORT[y_field[i]]}',
              color='green' if ok else 'red', fontsize=8.5)
plt.suptitle(f'{best} on real field photos (PlantDoc)'); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'field_example_predictions.png'), dpi=120); plt.show()

## ✅ What this tells us (discussion for the report)
- On **lab photos** (PlantVillage test set), accuracy is very high.
- On **field photos** (PlantDoc), accuracy drops a lot. Mohanty et al. (2016), our main reference, reported the same effect. Their model fell to about 31% on images from other sources.
- **Why?** PlantVillage photos show one leaf on a plain grey background under even lighting. Field photos have soil, hands, other leaves, shadows and blur that the model never saw during training. The model has partly learned the *photo style*, not only the disease. This is called **domain shift**.
- Field **top-3 accuracy** and **correct-crop %** are noticeably higher, so the model still gives useful hints.

### Future work (how to close the gap)
1. Train on mixed lab + field data (e.g. add part of PlantDoc to training).
2. Crop out just the leaf first (leaf detection/segmentation), then classify.
3. Stronger augmentation: random backgrounds, blur, lighting changes.
4. Collect real photos from local farms for the target crops.